# Decision Tree Analysis For GMS Prediction (Time-Binned Dataset)

In [8]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# Read dataset from CSV
path = "../data/time_binned_dataset.csv"
df = pd.read_csv(path)

# Select features (drop everything that could leak target, labels/IDs, timestamps), set target (forecast time), ignore missing values
x = df.drop(columns=[
    "ap_target_3h",
    "storm_3h",
    "ap_target_6h",
    "storm_6h",
    "ap_target_12h",
    "storm_12h",
    "ap_target_24h",
    "storm_24h",
    "datetime"
])

target1 = "storm_3h"; target2 = "storm_6h"; target3 = "storm_12h"; target4 = "storm_24h"
target = target4

y = df[target]

# Split data into training and testing
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

print(f"Training data shape: {x_train.shape}")
print(f"Testing data shape: {x_test.shape}")

# Account for class imbalance (rarity of gms)
model = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=0)

# Train
print("\nTraining the Decision Tree model...")
model.fit(x_train, y_train)

# Predict
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)[:, 1]

# Evaluate
print(f"\nClassification Report for {target[6:]} Prediction")
print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3)) # true positive rate vs false positive rate
print("PR-AUC :", round(average_precision_score(y_test, y_proba), 3)) # precision vs recall (for imbalanced data)
print("\nConfusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))

Training data shape: (70119, 49)
Testing data shape: (17530, 49)

Training the Decision Tree model...

Classification Report for 24h Prediction
              precision    recall  f1-score   support

           0      0.990     0.739     0.846     17219
           1      0.039     0.582     0.073       311

    accuracy                          0.736     17530
   macro avg      0.514     0.661     0.460     17530
weighted avg      0.973     0.736     0.833     17530

ROC-AUC: 0.724
PR-AUC : 0.046

Confusion matrix (rows=actual, cols=predicted):
[[12728  4491]
 [  130   181]]
